# SPY Weekly POC Core-Satellite Scan + Backtest

**The strategy (no options, just shares):**
- **2/3 of SPY just rides** — untouched, always invested, never capped.
- **1/3 trades around a line:** the line = the **most-crossed price of the last week** (the Point of
  Control / POC — the price SPY traded *through* the most). **Below the line → hold/own the 1/3.
  Above the line → sell the 1/3 to cash.** Sell-high / buy-low on a slice.

**Two parts:**
1. **THE MONDAY SCAN** — run it Monday morning. It draws the line over the last week (incl. today's
   open, so it catches the weekend gap), tells you where SPY sits vs the line, and gives the signal:
   *trim the third* / *hold* / *add the third back*.
2. **THE BACKTEST** — 15 years of SPY: how the weekly 2/3+1/3 version would have done vs just holding
   all of it, through the 2020 crash and 2022 bear, not only this bull.

---
*Honest frame (proven earlier): this is a **variance trade**, not a money-printer. Expect it to lag
3/3-hold slightly in a grind-up (the third misses a sliver of drift while parked in cash) and beat it
in chop / cushion it in a crash. The 2/3 core means it **cannot blow you up.** The backtest below tells
you the actual size of the drag and the cushion. Idle cash earns 0% here (conservative — real T-bills
would help the strategy). Taxes/wash-sales not modeled; run it in an IRA/Roth to avoid them.*


In [ ]:
# !pip install yfinance --quiet   # uncomment on a fresh runtime
import numpy as np, pandas as pd, yfinance as yf, matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')


## CONFIG


In [ ]:
TICKER      = 'SPY'
WINDOW      = 6        # sessions for the POC line (~last week incl. Monday). Widen to 10 for a steadier line.
N_LEVELS    = 250      # price-grid resolution for 'most-crossed' calc
CORE_FRAC   = 2/3      # the untouchable riding portion
REBAL_DAYS  = 5        # decision cadence in trading days (5 = weekly / Monday)
BT_YEARS    = 15       # backtest lookback (covers 2020 crash + 2022 bear + this bull)
CASH_YIELD  = 0.0      # annual yield on the parked 1/3 (0 = conservative; try 0.04 for T-bills)
START       = 10000
BAND_LOOKBACK = 40     # sessions (~8wks) to measure swing FREQUENCY over (stable estimate)
MIN_PER_WEEK  = 2.0    # reliability floor: a band must be hit at least this many days/week
THRESHOLDS    = [0.25,0.5,0.75,1.0,1.25,1.5,2.0,2.5,3.0]   # candidate trigger distances (%)


## The line: most-crossed price (Point of Control)


In [ ]:
def poc_line(highs, lows, n_levels=N_LEVELS):
    H = np.asarray(highs, float); L = np.asarray(lows, float)
    lo, hi = float(np.min(L)), float(np.max(H))
    if hi <= lo: return lo
    grid = np.linspace(lo, hi, n_levels)
    counts = np.zeros(n_levels)
    for i in range(len(H)):
        counts += ((grid >= L[i]) & (grid <= H[i])).astype(float)   # bar's range 'crosses' this price
    poc_levels = grid[counts == counts.max()]
    return float(np.median(poc_levels))                             # most-crossed price (median if tie)


In [ ]:
def reliable_band(hist, window, lookback, thresholds, min_per_week):
    '''Over the last `lookback` sessions, measure how far each day's low/high reached vs the rolling POC.
    Returns median reaches + a per-threshold days/week frequency table + the widest band still hit
    >= min_per_week on each side (the recommended add/trim trigger distances).'''
    H=hist['High'].values; L=hist['Low'].values; n=len(hist)
    downs=[]; ups=[]
    start=max(window, n-lookback)
    for i in range(start, n):
        line=poc_line(H[i-window+1:i+1], L[i-window+1:i+1])
        downs.append((line-L[i])/line*100)   # % the day's LOW went below the line
        ups.append((H[i]-line)/line*100)     # % the day's HIGH went above the line
    downs=np.array(downs); ups=np.array(ups)
    rows=[]
    for x in thresholds:
        rows.append(dict(pct=x,
                         days_below_per_wk=round((downs>=x).mean()*5,2),
                         days_above_per_wk=round((ups>=x).mean()*5,2)))
    tbl=pd.DataFrame(rows)
    ok_dn=[x for x in thresholds if (downs>=x).mean()*5 >= min_per_week]
    ok_up=[x for x in thresholds if (ups>=x).mean()*5   >= min_per_week]
    add_trig  = max(ok_dn) if ok_dn else None    # widest reliable DOWN band -> add trigger
    trim_trig = max(ok_up) if ok_up else None    # widest reliable UP band  -> trim trigger
    return dict(median_below=float(np.median(downs)), median_above=float(np.median(ups)),
                table=tbl, add_trig=add_trig, trim_trig=trim_trig)


## PART 1 — THE MONDAY SCAN  (run this Monday morning)


In [ ]:
scan = yf.download(TICKER, period='3mo', auto_adjust=True, progress=False)
if isinstance(scan.columns, pd.MultiIndex): scan.columns = scan.columns.get_level_values(0)
scan = scan.dropna()
win = scan.tail(WINDOW)
line = poc_line(win['High'].values, win['Low'].values)
price = float(scan['Close'].iloc[-1])
asof  = scan.index[-1].date()
dist  = (price/line - 1)*100
above = price > line

print('='*54)
print(f'  {TICKER} WEEKLY POC SCAN   (as of {asof})')
print('='*54)
print(f'  Most-crossed line (last {WINDOW} sessions): ${line:,.2f}')
print(f'  SPY now                              : ${price:,.2f}')
print(f'  Distance                             : {dist:+.2f}%')
print('-'*54)
if above:
    print('  SIGNAL: ABOVE the line  ->  the 1/3 should be in CASH')
    print('          (if you are holding the third, TRIM it here)')
else:
    print('  SIGNAL: BELOW the line  ->  the 1/3 should be INVESTED')
    print('          (if the third is in cash, ADD it back here)')
print('  The 2/3 core rides regardless.')
print('='*54)

band = reliable_band(scan, WINDOW, BAND_LOOKBACK, THRESHOLDS, MIN_PER_WEEK)
print()
print('  RELIABLE SWING BAND (what to aim for, last ~%dwk):' % (BAND_LOOKBACK//5))
print(f'    typical daily reach: {band["median_below"]:.2f}% below / {band["median_above"]:.2f}% above the line')
at = band['add_trig']; tt = band['trim_trig']
if at: print(f'    ADD  the 1/3 when SPY <= ${line*(1-at/100):,.2f}  (-{at:.2f}%, hit >= {MIN_PER_WEEK:g}x/wk)')
else:  print('    ADD  trigger: no down-band reaches the 2x/wk floor -> market too quiet/one-way')
if tt: print(f'    TRIM the 1/3 when SPY >= ${line*(1+tt/100):,.2f}  (+{tt:.2f}%, hit >= {MIN_PER_WEEK:g}x/wk)')
else:  print('    TRIM trigger: no up-band reaches the 2x/wk floor')
print('='*54)


### How to read the scan
- **Above the line** = SPY is extended vs where it traded all week → your satellite third belongs in cash.
- **Below the line** = SPY is cheap vs its weekly center → your third belongs invested.
- You only *act* when the signal flips from last week. Same signal = do nothing.
- The 2/3 core never moves.


## Part 1.5 — Trigger calibration table (the full picture)


In [ ]:
b = reliable_band(scan, WINDOW, BAND_LOOKBACK, THRESHOLDS, MIN_PER_WEEK)
print(f'How many days/week SPY reached each distance from the line (last ~{BAND_LOOKBACK//5}wk):')
print(b['table'].to_string(index=False))
print(f"\nWidest band hit >= {MIN_PER_WEEK:g}x/week -> ADD at -{b['add_trig']}% , TRIM at +{b['trim_trig']}%" )
print('Set your triggers there: tight enough to fill ~twice a week, wide enough to skip the noise.')


## PART 2 — THE BACKTEST  (does the overlay earn its keep?)


In [ ]:
def max_dd(s):
    return float((s/s.cummax()-1).min()*100)

def run_backtest(hist, window, rebal_days, core_frac, start, cash_yield):
    hist = hist.dropna()
    px = hist['Close'].values; H = hist['High'].values; L = hist['Low'].values; n = len(px)
    daily_cash_g = (1+cash_yield)**(1/252)
    core_units = (start*core_frac)/px[0]
    sat_units  = (start*(1-core_frac))/px[0]; sat_cash = 0.0; sat_in = True
    equity=[]; trades=0; sat_days_in=0
    for i in range(n):
        if sat_cash>0: sat_cash *= daily_cash_g
        if i>=window and i % rebal_days == 0:
            line = poc_line(H[i-window+1:i+1], L[i-window+1:i+1])
            want_in = px[i] < line                # below line -> invested; above -> cash
            if want_in and not sat_in:
                sat_units = sat_cash/px[i]; sat_cash=0.0; sat_in=True; trades+=1
            elif (not want_in) and sat_in:
                sat_cash = sat_units*px[i]; sat_units=0.0; sat_in=False; trades+=1
        if sat_in: sat_days_in += 1
        equity.append(core_units*px[i] + sat_units*px[i] + sat_cash)
    eq = pd.Series(equity, index=hist.index)
    hold = pd.Series(px/px[0]*start, index=hist.index)
    return eq, hold, trades, sat_days_in, n

hist = yf.download(TICKER, period=f'{BT_YEARS}y', auto_adjust=True, progress=False)
if isinstance(hist.columns, pd.MultiIndex): hist.columns = hist.columns.get_level_values(0)
eq, hold, trades, sat_in_days, n = run_backtest(hist, WINDOW, REBAL_DAYS, CORE_FRAC, START, CASH_YIELD)

yrs = n/252
def cagr(s): return ((s.iloc[-1]/s.iloc[0])**(1/yrs)-1)*100
print(f'--- {TICKER} weekly 2/3+1/3 POC overlay vs 3/3 buy-and-hold  ({yrs:.1f}y, window={WINDOW}) ---')
print(f'  Strategy : {(eq.iloc[-1]/START-1)*100:8.1f}%  total | CAGR {cagr(eq):5.2f}% | maxDD {max_dd(eq):6.1f}%')
print(f'  Buy&Hold : {(hold.iloc[-1]/START-1)*100:8.1f}%  total | CAGR {cagr(hold):5.2f}% | maxDD {max_dd(hold):6.1f}%')
print(f'  Diff     : {(eq.iloc[-1]-hold.iloc[-1])/START*100:8.1f}%  total | {cagr(eq)-cagr(hold):+5.2f}% CAGR')
print(f'  Trades: {trades} over {yrs:.1f}y (~{trades/yrs:.1f}/yr) | 1/3 invested {sat_in_days/n*100:.0f}% of days')
plt.figure(figsize=(11,5))
plt.plot(hold.index, hold.values, label=f'3/3 Buy & Hold  (maxDD {max_dd(hold):.0f}%)', lw=1.6)
plt.plot(eq.index, eq.values, label=f'2/3 + 1/3 POC  (maxDD {max_dd(eq):.0f}%)', lw=1.6)
plt.title(f'{TICKER}: weekly POC core-satellite vs buy-and-hold'); plt.legend(); plt.grid(alpha=0.3); plt.yscale('log'); plt.show()


### The read
- **Diff on total return** = the cost (or gain) of the overlay. Expect a small *negative* over a 15y
  span dominated by bull years — the third misses drift while parked in cash.
- **maxDD** = the payoff. Expect the overlay's max drawdown to be *meaningfully smaller* — that's the
  crash cushion you're buying.
- **Trades/yr** should be low (weekly cadence = a handful of flips a year = low friction/tax).
- If you want to see the *chop* case where the overlay shines, re-run with `BT_YEARS = 3` starting from
  a sideways stretch, or set `CASH_YIELD = 0.04` to credit the parked third with T-bills — both tilt the
  comparison toward the strategy and show you the real-world picture.

**Bottom line to look for:** if the overlay gives up ~0.5-1%/yr of CAGR to cut max drawdown by a third or
more, that's the trade — you're paying a little return for a lot less pain, and the 2/3 core guarantees
you can't be wrecked. If it gives up more than that for little drawdown benefit, the POC line isn't
earning its keep and plain 3/3-hold wins.
